# Task B -- TAPT on every comment, then five seeds on every row

The same recipe as `f_tapt`, with one change: the adaptation pass now reads **all** of its
corpus. `f_tapt` adapted MuRIL on 95% of it, because `tapt.py` held 5% back to print
perplexity and also dropped one-word comments.

| stage | `f_tapt` (last run) | this notebook |
|---|---|---|
| TAPT text | 6,044 of 6,406 comments | **all 6,406** |
| classifier rows per model | 3,143 | 3,143 |
| seeds, averaged | 42 43 44 45 46 | 42 43 44 45 46 |
| epochs | 6 | 6 |

The TAPT corpus is `multiclass_train.csv` (3,159 comments) plus the external Kannada
OffensEval corpus (3,247). No labels are read at this stage.

The classifier trains on 3,143 rows because deduplication drops 16: 12 repeated copies of
comments that stay in, and the 4 rows of 2 comments whose copies were given different
labels. That matches `f_tapt`, so the TAPT corpus is the only change between the two.

**There is no local score, and there cannot be one.** Every labelled row is in training.
The only way to score this is to upload `b_tapt_full.zip` to CodaBench.

About 2 hours on a T4. Sidebar: Accelerator `GPU T4 x2` or `GPU P100`, Internet on.
Save Version -> Save & Run All.

In [ ]:
import os, re, subprocess, sys, pathlib
WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git","-C",WORK,"pull","--ff-only"], check=True)
else:
    subprocess.run(["git","clone","-q","-b","task-b","--depth","1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK); print("cwd:", os.getcwd())
subprocess.run(["git","log","-1","--oneline"], check=True)
assert "--min-words" in pathlib.Path("work/tapt.py").read_text(), \
    "this clone of task-b predates the full-corpus TAPT flags; push them first"
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    """Streams, tees, and raises. Nothing here is allowed to fail quietly."""
    print(f"$ {' '.join(cmd)}", flush=True)
    fh = open(log,"w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh: fh.write(line)
    p.wait()
    if fh: fh.close()
    if p.returncode: raise RuntimeError(f"exit {p.returncode}: {cmd}")

## 1. TAPT on every comment (~25 min)

`--val-frac 0` holds nothing back, `--min-words 1` keeps one-word comments and
`--no-dedupe` keeps repeated ones. The cell after it reads the log and stops the notebook
if anything was left out.

In [ ]:
run([sys.executable,"-u","work/tapt.py","--out","work/runs/tapt-full",
     "--val-frac","0","--min-words","1","--no-dedupe"],
    log="work/tapt_full.log")

In [ ]:
t = pathlib.Path("work/tapt_full.log").read_text()
m = re.search(r"MLM trains on (\d+) of (\d+) comments, (\d+) held out", t)
assert m, "could not find the corpus line in the TAPT log"
used, total, held = map(int, m.groups())
print(f"TAPT trained on {used} of {total} comments, {held} held out")
assert used == total and held == 0, "TAPT did not use the whole corpus"
assert total == 6406, f"expected 6,406 comments (3,159 + 3,247), got {total}"

## 2. Five seeds, every row (~95 min)

`--folds 1` trains each seed on all 3,143 rows and keeps its final checkpoint. The five
models' test probabilities are averaged.

In [ ]:
run([sys.executable,"-u","work/muril_b.py","--tag","b_tapt_full",
     "--model","work/runs/tapt-full","--folds","1",
     "--seeds","42","43","44","45","46","--epochs","6"],
    log="work/b_tapt_full.log")

In [ ]:
t = pathlib.Path("work/b_tapt_full.log").read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", t)
for seed, rows in fits:
    print(f"seed {seed}: trained on {rows} rows")
assert [s for s, _ in fits] == ["42","43","44","45","46"], "not all five seeds ran"
assert all(r == "3143" for _, r in fits), "a seed trained on fewer than 3,143 rows"

## 3. Package

The agreement figure is how often this run's 395 predictions match `b_tapt_5f`, which
scored 0.5922 on CodaBench. It is not a score.

In [ ]:
import pandas as pd
ZIP = "/kaggle/working/b_tapt_full.zip"
run([sys.executable,"work/make_submission.py","--task","b",
     "--pred","work/runs/b_tapt_full/predictions.csv","--out",ZIP])
for f in ("work/tapt_full.log","work/b_tapt_full.log"):
    run(["cp",f,"/kaggle/working/"])
run(["cp","work/runs/b_tapt_full/predictions.csv","/kaggle/working/b_tapt_full_predictions.csv"])
run(["unzip","-l",ZIP])

new = pd.read_csv("work/runs/b_tapt_full/predictions.csv").set_index("id")["label"]
ref = pd.read_csv("submissions/b_tapt_5f/predictions.csv").set_index("id")["label"]
print(f"\nagrees with b_tapt_5f on {100*(new.reindex(ref.index) == ref).mean():.1f}% of test rows")
print("\npredicted distribution:")
print((100*new.value_counts(normalize=True)).round(1).to_string())
print("\nDownload b_tapt_full.zip from the Output tab and upload it to the Task B phase.")